# Raw VD Query Speed Demo

Interactive Colab demo for comparing DuckDB + raw Parquet on Google Drive with pandas `read_csv` over a same-day CSV cache. Choose an accident preset or enter a manual route/time/window; SQL is generated inside `scripts/demo/raw_vd_query_benchmark.py`.

In [ ]:
import os, sys
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    _DEFAULT_ROOT = '/content/drive/MyDrive/陽交大高速公路計畫/main'
else:
    _here = Path(os.getcwd()).resolve()
    for _p in [_here, *_here.parents]:
        if (_p / 'CLAUDE.md').exists():
            _DEFAULT_ROOT = str(_p)
            break
    else:
        _DEFAULT_ROOT = str(_here)

_candidates = [Path(os.environ['PROJECT_ROOT'])] if 'PROJECT_ROOT' in os.environ else []
_candidates.append(Path(_DEFAULT_ROOT))
_candidates.extend([Path(os.getcwd()).resolve(), *Path(os.getcwd()).resolve().parents])

PROJECT_ROOT = None
for _root in _candidates:
    if (_root / 'scripts' / 'demo' / 'raw_vd_query_benchmark.py').exists():
        PROJECT_ROOT = str(_root)
        break
    if (_root / 'CLAUDE.md').exists() and (_root / 'scripts').exists():
        PROJECT_ROOT = str(_root)
        break

if PROJECT_ROOT is None:
    checked = '\n'.join(f'  - {_p}' for _p in _candidates[:8])
    raise FileNotFoundError(
        'Could not find the project root containing scripts/demo/raw_vd_query_benchmark.py. '\
        'In Colab, confirm Drive has synced the new scripts/demo folder. Checked:\n' + checked
    )

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
%cd {PROJECT_ROOT}

%pip install -q duckdb pandas pyarrow plotly ipywidgets

if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()

import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output

import importlib
importlib.invalidate_caches()
try:
    from scripts.demo.raw_vd_query_benchmark import (
        DEFAULT_CACHE_DIR,
        RawQuerySpec,
        prepare_csv_cache,
        run_benchmark,
    )
except ModuleNotFoundError as exc:
    for _name in list(sys.modules):
        if _name == 'scripts' or _name.startswith('scripts.'):
            sys.modules.pop(_name, None)
    importlib.invalidate_caches()
    try:
        from scripts.demo.raw_vd_query_benchmark import (
            DEFAULT_CACHE_DIR,
            RawQuerySpec,
            prepare_csv_cache,
            run_benchmark,
        )
    except ModuleNotFoundError as exc2:
        demo_path = Path(PROJECT_ROOT) / 'scripts' / 'demo' / 'raw_vd_query_benchmark.py'
        raise ModuleNotFoundError(
            f'Could not import scripts.demo.raw_vd_query_benchmark from PROJECT_ROOT={PROJECT_ROOT}. '\
            f'Expected file: {demo_path}. Exists={demo_path.exists()}. '\
            'Restart the runtime or rerun this setup cell after Drive sync completes.'
        ) from exc2
importlib.invalidate_caches()
try:
    from scripts.lib.data_loaders import read_accidents
except ImportError:
    for _name in list(sys.modules):
        if _name == 'scripts.lib' or _name.startswith('scripts.lib.'):
            sys.modules.pop(_name, None)
    importlib.invalidate_caches()
    from scripts.lib.data_loaders import read_accidents

DATA_ROOT = Path(PROJECT_ROOT) / 'data'
CACHE_DIR = DEFAULT_CACHE_DIR
print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATA_ROOT    =', DATA_ROOT)
print('CACHE_DIR   =', CACHE_DIR)

## Widget Demo

In [ ]:
accidents = read_accidents(data_root=DATA_ROOT)
accidents = accidents.dropna(subset=['accident_id', 'freeway', 'direction', 'mileage_m', 'report_time']).copy()
accidents = accidents[accidents['direction'].isin(['N', 'S', 'E', 'W'])].sort_values(['report_time', 'accident_id']).reset_index(drop=True)

def accident_label(row):
    ts = pd.Timestamp(row['report_time']).strftime('%Y-%m-%d %H:%M')
    return f"{row['accident_id']} | {ts} | {row['freeway']}-{row['direction']} {float(row['mileage_km']):.1f}K"

accident_options = [(accident_label(row), str(row['accident_id'])) for _, row in accidents.iterrows()]
first = accidents.iloc[0]

mode_w = widgets.ToggleButtons(options=['accident preset', 'manual input'], value='accident preset', description='Mode:')
accident_w = widgets.Dropdown(options=accident_options, value=str(first['accident_id']), description='Accident:', layout=widgets.Layout(width='760px'), style={'description_width': 'initial'})
date_w = widgets.Text(value=pd.Timestamp(first['report_time']).strftime('%Y-%m-%d'), description='Date:', layout=widgets.Layout(width='180px'))
time_w = widgets.Text(value=pd.Timestamp(first['report_time']).strftime('%H:%M'), description='Time:', layout=widgets.Layout(width='160px'))
freeway_w = widgets.Dropdown(options=sorted(accidents['freeway'].dropna().unique().tolist()), value=str(first['freeway']), description='Freeway:', layout=widgets.Layout(width='180px'))
direction_w = widgets.Dropdown(options=['N', 'S', 'E', 'W'], value=str(first['direction']), description='Direction:', layout=widgets.Layout(width='180px'))
mileage_w = widgets.FloatText(value=float(first['mileage_km']), description='Mileage km:', layout=widgets.Layout(width='220px'), style={'description_width': 'initial'})
upstream_w = widgets.IntSlider(value=50, min=0, max=80, step=1, description='Upstream km:', continuous_update=False, layout=widgets.Layout(width='420px'), style={'description_width': 'initial'})
downstream_w = widgets.IntSlider(value=10, min=0, max=30, step=1, description='Downstream km:', continuous_update=False, layout=widgets.Layout(width='420px'), style={'description_width': 'initial'})
before_w = widgets.IntSlider(value=60, min=0, max=180, step=5, description='Before min:', continuous_update=False, layout=widgets.Layout(width='420px'), style={'description_width': 'initial'})
after_w = widgets.IntSlider(value=240, min=0, max=360, step=5, description='After min:', continuous_update=False, layout=widgets.Layout(width='420px'), style={'description_width': 'initial'})
repeat_w = widgets.IntSlider(value=3, min=1, max=10, step=1, description='Repeat:', continuous_update=False, layout=widgets.Layout(width='300px'))

status_w = widgets.HTML(value='<b>Status:</b> ready')
prepare_btn = widgets.Button(description='Prepare CSV cache', button_style='warning', icon='database')
run_btn = widgets.Button(description='Run benchmark', button_style='primary', icon='play')
render_btn = widgets.Button(description='Render preview / heatmap', button_style='info', icon='bar-chart')
out = widgets.Output()
last_result = {'value': None}

def load_selected_accident(_change=None):
    row = accidents[accidents['accident_id'].astype(str) == str(accident_w.value)].iloc[0]
    ts = pd.Timestamp(row['report_time'])
    date_w.value = ts.strftime('%Y-%m-%d')
    time_w.value = ts.strftime('%H:%M')
    freeway_w.value = str(row['freeway'])
    direction_w.value = str(row['direction'])
    mileage_w.value = float(row['mileage_km'])

accident_w.observe(load_selected_accident, names='value')

def current_spec():
    if mode_w.value == 'accident preset':
        load_selected_accident()
    return RawQuerySpec(
        date=date_w.value.strip(),
        time=time_w.value.strip(),
        freeway=str(freeway_w.value),
        direction=str(direction_w.value),
        mileage_km=float(mileage_w.value),
        upstream_km=int(upstream_w.value),
        downstream_km=int(downstream_w.value),
        before_min=int(before_w.value),
        after_min=int(after_w.value),
        data_root=DATA_ROOT,
    )

def show_spec(spec):
    print(f'Time window:    {spec.start_time} -> {spec.end_time}')
    print(f'Route/window:   {spec.freeway}-{spec.direction}, {spec.mileage_bounds_m[0] / 1000:.3f}K -> {spec.mileage_bounds_m[1] / 1000:.3f}K')
    print('Raw parquet:')
    for path in spec.source_paths():
        print(f'  - {path} ({path.stat().st_size / 1024 / 1024:.2f} MiB)')

def _show_error(label, exc):
    status_w.value = f'<b>Status:</b> {label} failed'
    with out:
        clear_output(wait=True)
        print(f'{label} failed: {type(exc).__name__}: {exc}')
        import traceback
        traceback.print_exc()

def on_prepare(_button):
    status_w.value = '<b>Status:</b> preparing CSV cache...'
    prepare_btn.disabled = True
    try:
        with out:
            clear_output(wait=True)
            spec = current_spec()
            show_spec(spec)
            csv_paths, seconds = prepare_csv_cache(spec.source_paths(), cache_dir=CACHE_DIR)
            print(f'CSV cache prepare seconds: {seconds:.3f}')
            for path in csv_paths:
                print(f'  - {path} ({path.stat().st_size / 1024 / 1024:.2f} MiB)')
        status_w.value = '<b>Status:</b> CSV cache ready'
    except Exception as exc:
        _show_error('Prepare CSV cache', exc)
    finally:
        prepare_btn.disabled = False

def on_run(_button):
    status_w.value = '<b>Status:</b> running benchmark...'
    run_btn.disabled = True
    try:
        with out:
            clear_output(wait=True)
            spec = current_spec()
            show_spec(spec)
            result = run_benchmark(spec, repeat=int(repeat_w.value), prepare_csv=False, cache_dir=CACHE_DIR)
            last_result['value'] = result
            display(result['summary'])
            duck = result['summary'].set_index('engine').loc['DuckDB + Parquet']
            csv = result['summary'].set_index('engine').loc['pandas read_csv + filter']
            if duck['station_rows'] != csv['station_rows'] or duck['raw_lane_rows'] != csv['raw_lane_rows']:
                print('WARNING: result row counts differ between engines.')
            print('Run complete. Click Render preview / heatmap to inspect the result.')
        status_w.value = '<b>Status:</b> benchmark complete'
    except Exception as exc:
        _show_error('Run benchmark', exc)
    finally:
        run_btn.disabled = False

def on_render(_button):
    status_w.value = '<b>Status:</b> rendering preview...'
    render_btn.disabled = True
    try:
        with out:
            clear_output(wait=True)
            result = last_result['value']
            if result is None:
                print('Run benchmark first.')
                status_w.value = '<b>Status:</b> run benchmark first'
                return
            frame = result['duckdb_frame'].copy()
            display(result['summary'])
            display(frame.head(100))
            if frame.empty:
                print('No rows in this query window.')
                status_w.value = '<b>Status:</b> no rows to render'
                return
            frame['mileage_km'] = frame['mileage_m'] / 1000.0
            pivot = frame.pivot_table(index='mileage_km', columns='timestamp', values='speed', aggfunc='mean').sort_index()
            fig = px.imshow(
                pivot,
                aspect='auto',
                origin='lower',
                color_continuous_scale='RdYlGn',
                labels={'x': 'timestamp', 'y': 'mileage_km', 'color': 'speed km/h'},
                title='Raw VD stationized speed heatmap from DuckDB result',
            )
            fig.update_layout(height=560, margin=dict(l=50, r=20, t=60, b=60))
            fig.show()
        status_w.value = '<b>Status:</b> preview rendered'
    except Exception as exc:
        _show_error('Render preview', exc)
    finally:
        render_btn.disabled = False

# Deprecated duplicate handlers are intentionally shadowed below by the robust handlers above.
def _unused_old_handlers_marker():
    return None

'''
Old unguarded callback bodies retained only as disabled source text in this cell.
They are not executed.

def on_prepare(_button):
    with out:
        clear_output(wait=True)
        spec = current_spec()
        show_spec(spec)
        csv_paths, seconds = prepare_csv_cache(spec.source_paths(), cache_dir=CACHE_DIR)
        print(f'CSV cache prepare seconds: {seconds:.3f}')
        for path in csv_paths:
            print(f'  - {path} ({path.stat().st_size / 1024 / 1024:.2f} MiB)')

def on_run(_button):
    with out:
        clear_output(wait=True)
        spec = current_spec()
        show_spec(spec)
        result = run_benchmark(spec, repeat=int(repeat_w.value), prepare_csv=False, cache_dir=CACHE_DIR)
        last_result['value'] = result
        display(result['summary'])
        duck = result['summary'].set_index('engine').loc['DuckDB + Parquet']
        csv = result['summary'].set_index('engine').loc['pandas read_csv + filter']
        if duck['station_rows'] != csv['station_rows'] or duck['raw_lane_rows'] != csv['raw_lane_rows']:
            print('WARNING: result row counts differ between engines.')
        print('Run complete. Click Render preview / heatmap to inspect the result.')

def on_render(_button):
    with out:
        clear_output(wait=True)
        result = last_result['value']
        if result is None:
            print('Run benchmark first.')
            return
        frame = result['duckdb_frame'].copy()
        display(result['summary'])
        display(frame.head(100))
        if frame.empty:
            print('No rows in this query window.')
            return
        frame['mileage_km'] = frame['mileage_m'] / 1000.0
        pivot = frame.pivot_table(index='mileage_km', columns='timestamp', values='speed', aggfunc='mean').sort_index()
        fig = px.imshow(
            pivot,
            aspect='auto',
            origin='lower',
            color_continuous_scale='RdYlGn',
            labels={'x': 'timestamp', 'y': 'mileage_km', 'color': 'speed km/h'},
            title='Raw VD stationized speed heatmap from DuckDB result',
        )
        fig.update_layout(height=560, margin=dict(l=50, r=20, t=60, b=60))
        fig.show()
'''

prepare_btn.on_click(on_prepare)
run_btn.on_click(on_run)
render_btn.on_click(on_render)

controls = widgets.VBox([
    mode_w,
    accident_w,
    widgets.HBox([date_w, time_w, freeway_w, direction_w, mileage_w]),
    widgets.HBox([upstream_w, downstream_w]),
    widgets.HBox([before_w, after_w, repeat_w]),
    status_w,
    widgets.HBox([prepare_btn, run_btn, render_btn]),
])
display(controls, out)
load_selected_accident()